In [1]:
import sys
import os
import torch

current_dir = os.path.dirname(os.path.abspath('/mnt/workspace/_/simpleRL-reason/evaluation/eval_logic/test_eval.ipynb'))
root_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(root_dir)
from framework.register import register_processor
from utils.util import load_json, save_json, timestamped_print
from infer_module.infer_reward import Reward_Service


/cpfs02/user/liurunze/miniforge3/envs/zj_or1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
from infer_module import infer_reward as infer_critic_module
import importlib
importlib.reload(infer_critic_module)

<module 'infer_module.infer_reward' from '/mnt/workspace/_/simpleRL-reason/evaluation/infer_module/infer_reward.py'>

In [15]:
from infer_module.infer_reward import Reward_Service

In [16]:
_cached_reward_service = None  # Global variable to cache the reward_Service instance

def get_reward_service(model_path: str = None, tensor_parallel_size: int = 1) -> Reward_Service:
    global _cached_reward_service
    if _cached_reward_service is None:
        timestamped_print("REWARD: Initializing reward service...")
        try:
            _cached_reward_service = Reward_Service(
                model_path=model_path,
                tensor_parallel_size=tensor_parallel_size,
                good_token='+',
                bad_token='-',
                step_tag='ки'
            )
        except Exception as e:
            timestamped_print(f"REWARD: Failed to initialize: {e}", "ERROR")
            raise
    return _cached_reward_service

In [3]:
class Args:
    def __init__(self):
        self.input_filepath = '/mnt/workspace/_/simpleRL-reason/_outputs/policy_test/7B_ppo/math500/0.json'
        self.output_filepath = '/mnt/workspace/_/simpleRL-reason/_outputs/policy_test/7B_ppo_reward/math500/0.json'
        self.reward_path = '/mnt/workspace/hf_models/models--peiyi9979--math-shepherd-mistral-7b-prm'
        self.reward_tensor_parallel_size = 1
        self.system_prompt = "You are a helpful assistant."
        self.user_prompt_template = "{problem}\nPlease reason step by step, and put your final answer within \\boxed{{}}."
        
args = Args()

In [23]:
data = load_json(args.input_filepath)
reward_service = get_reward_service(model_path=args.reward_path, tensor_parallel_size=args.reward_tensor_parallel_size)
data['values'] = []
data['tag_indices'] = []
for idd in range(len(data['policy_responses'])):
    messages = [ 
        { "role": "system", "content": args.system_prompt }, 
        { "role": "user", "content": args.user_prompt_template.format(problem=data['problem']) },
        { "role": "assistant", "content": data['policy_responses'][idd] }, 
    ]
    results = reward_service.build_prompt(messages)
    values = []
    tag_indices = []
    for prompt, response_length in results:
        tokens = reward_service.simple_tokenize(prompt)
        tag_indices.append(len(tokens))
        pass

    values, step_values = reward_service.predict_values(prompt, response_length)
    data['values'].append(values)
    data['tag_indices'].append(tag_indices)

save_json(data, args.output_filepath)

[2025-05-22 06:52:06] [INFO] 	UTILS: Successfully loaded JSON from /mnt/workspace/_/simpleRL-reason/_outputs/policy_test/7B_ppo/math500/0.json
[2025-05-22 06:52:09] [INFO] 	UTILS: Successfully saved JSON to /mnt/workspace/_/simpleRL-reason/_outputs/policy_test/7B_ppo_reward/math500/0.json


In [50]:
tag_token_id = critic_service.tokenizer.encode("]]\n\n", add_special_tokens=False)
m = len(tag_token_id)
tag_token_id

[42450]

In [58]:
len(values)

336

In [59]:
prompt

'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nConvert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$\nPlease reason step by step, and put your final answer within \\boxed{}.<|im_end|>\n<|im_start|>assistant\nTo convert the point \\((0, 3)\\) from rectangular coordinates to polar coordinates, we need to find the values of \\(r\\) and \\(\\theta\\). The formulas for converting from rectangular coordinates \\((x, y)\\) to polar coordinates \\((r, \\theta)\\) are:\n\\[\nr = \\sqrt{x^2 + y^2}\n\\]\n\\[\n\\theta = \\tan^{-1}\\left(\\frac{y}{x}\\right)\n\\]\nGiven the point \\((0, 3)\\), we identify \\(x = 0\\) and \\(y = 3\\).\n\nFirst, we calculate \\(r\\):\n\\[\nr = \\sqrt{0^2 + 3^2} = \\sqrt{9} = 3\n\\]\n\nNext, we calculate \\(\\theta\\). The formula \\(\\theta = \\tan^{-1}\\left(\\frac{y}{x}\\right)\\) results in an indeterminate form \\(\\tan^{

In [19]:
sum(tag_masks)

2

In [61]:
tag_indices

[127, 163, 253, 302, 336]

In [26]:
response_length = len(response_length)

In [7]:
del Critic_Service

In [8]:
from infer_module.infer_critic import Critic_Service

In [29]:
Critic_Service.get_symbol_set(critic_service, critic_service.tokenizer, "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~")

{(5, 271),
 (27, 271),
 (58, 271),
 (59, 271),
 (61, 271),
 (271,),
 (382,),
 (401,),
 (630,),
 (692,),
 (1339,),
 (1447,),
 (1837,),
 (1939,),
 (2219,),
 (2533,),
 (3554,),
 (3876,),
 (4257,),
 (8680,),
 (10452,),
 (15441,),
 (18797,),
 (19324,),
 (19347,),
 (21518,),
 (26487,),
 (41025,),
 (43738,),
 (44611,),
 (58629,),
 (66426,),
 (68327,)}

In [62]:
from transformers import AutoConfig, AutoModelForTokenClassification

In [ ]:
local_path = "/cpfs02/user/liurunze/_/simpleRL-reason/_outputs/checkpoints/verl-ppo_models--Qwen--Qwen2.5-7B_simplelr_qwen_level3to5_max_response8192_batch1024_rollout8_klcoef0.0001_entcoef0.001/global_step_65/critic/huggingface"
critic_module = AutoModelForTokenClassification.from_pretrained(pretrained_model_name_or_path=local_path,
                                                                            torch_dtype=torch_dtype,
                                                                            config=critic_model_config,
                                                                            attn_implementation='flash_attention_2',
                                                                            trust_remote_code=trust_remote_code)